# 04 - Network Analysis

Build a combined conceptual network from policy + Yelp data, apply SNA techniques, cross-source analysis, and generate visualisations.

In [ ]:
import sys
sys.path.insert(0, '..')

from src.ingest.pdf_reader import read_policy_pdf, pages_to_document
from src.ingest.yelp_reader import load_yelp_reviews
from src.preprocessing.cleaner import clean_text
from src.preprocessing.tokeniser import SpacyTokeniser
from src.extraction.concept_extractor import ConceptExtractor
from src.extraction.relationship_extractor import RelationshipExtractor
from src.network.graph_builder import GraphBuilder
from src.network.graph_analysis import GraphAnalyser
from src.visualisation.static_viz import StaticVisualiser
from src.visualisation.interactive_viz import InteractiveVisualiser

## 1. Ingest, Preprocess, Extract, Build Network (Combined)

In [ ]:
tokeniser = SpacyTokeniser()
documents = []

# Policy
pages = read_policy_pdf()
cleaned_policy = clean_text(pages_to_document(pages), source='policy')
documents.append(tokeniser.process(cleaned_policy, doc_id='policy', source='policy'))
print(f'Policy: {len(documents[0].sentences)} sentences')

# Yelp (100 reviews for demo speed)
df = load_yelp_reviews(category_filter='Restaurants', sample_n=100)
yelp_texts = [(row['review_id'], row['text']) for _, row in df.iterrows()]
cleaned_yelp = [(did, clean_text(t, source='yelp')) for did, t in yelp_texts]
cleaned_yelp = [(did, t) for did, t in cleaned_yelp if t.strip()]
documents.extend(tokeniser.process_batch(cleaned_yelp, source='yelp'))
print(f'Combined: {len(documents)} documents')

# Extract and build
concepts = ConceptExtractor().extract(documents)
relationships = RelationshipExtractor().extract(documents, concepts)
G = GraphBuilder(min_edge_weight=2).build(concepts, relationships)
print(f'Network: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

## 2. Centrality Analysis

In [ ]:
analyser = GraphAnalyser(G)
centrality_df = analyser.all_centralities()
centrality_df.head(15)

## 3. Community Detection

In [ ]:
try:
    partition = analyser.detect_communities_louvain()
except ImportError:
    partition = analyser.detect_communities_label_propagation()

comm_df = analyser.community_summary(partition)
print(f'{len(comm_df)} communities detected')
comm_df

## 4. Brokerage Analysis

In [ ]:
brokers_df = analyser.find_brokers(partition)
print('Top brokers (bridge concepts between communities):')
brokers_df

## 5. Cross-Source Analysis

Identify concepts and relationships that bridge the policy and Yelp domains.

In [ ]:
src_dist = analyser.source_distribution()
print('Concept counts by source type:')
print(src_dist.to_string(index=False))

comparison = analyser.source_comparison_summary()
print(f'\nPolicy-only: {comparison["policy_only_concepts"]}')
print(f'Yelp-only:   {comparison["yelp_only_concepts"]}')
print(f'Shared:      {comparison["shared_concepts"]} ({comparison["overlap_pct"]:.1f}%)')

bridges = analyser.bridging_concepts(top_n=15)
print('\nTop bridging concepts:')
bridges[['label', 'source_type', 'bridge_score', 'cross_source_neighbors']].head(10)

## 6. Visualisations

In [ ]:
G = analyser.annotate_graph(partition)

viz = StaticVisualiser('../results')
viz.plot_network_by_source(G, title='Combined Network — Coloured by Source')
viz.plot_source_overlap(G)
viz.plot_network(G, partition=partition, title='Combined Conceptual Network (by Community)')
viz.plot_top_concepts(centrality_df)
viz.plot_community_distribution(partition)
viz.plot_cooccurrence_heatmap(G)
print('Static plots saved to results/')

In [ ]:
iv = InteractiveVisualiser('../results')
path = iv.create_interactive_network(G, partition=partition,
    title='Combined Network Explorer', colour_by_source=True)
print(f'Interactive network saved to: {path}')